# Mixed Precision — FP32, FP16, BF16, and FP8 Explained

**Module 29** of the PyTorch Complete Learning Guide

This notebook explores numerical precision formats in PyTorch and how to use mixed precision for faster, more memory-efficient training.

**Topics:**
- Floating-point format anatomy (FP32, FP16, BF16, FP8)
- Automatic Mixed Precision (AMP) with `torch.amp.autocast`
- GradScaler for FP16 training
- BF16 vs FP16 tradeoffs
- FP8 scaled matmul concepts
- FSDP2 mixed precision patterns

**Prerequisites:** Module 07 (Training), Module 08 (torch.compile), Module 20 (Backends Tuning)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Numerical Formats Overview

Every floating-point number: `(-1)^sign × 2^exponent × (1 + mantissa)`

| Format | Bits | Exponent | Mantissa | Range | Precision | PyTorch dtype |
|--------|------|----------|----------|-------|-----------|---------------|
| FP32 | 32 | 8 | 23 | ±3.4e38 | High | `torch.float32` |
| TF32 | 19 | 8 | 10 | ±3.4e38 | Medium | (internal) |
| BF16 | 16 | 8 | 7 | ±3.4e38 | Low | `torch.bfloat16` |
| FP16 | 16 | 5 | 10 | ±65504 | Medium-low | `torch.float16` |
| FP8 E4M3 | 8 | 4 | 3 | ±448 | Very low | `torch.float8_e4m3fn` |
| FP8 E5M2 | 8 | 5 | 2 | ±57344 | Lowest | `torch.float8_e5m2` |

**Key tradeoff:** More exponent bits = larger range. More mantissa bits = more precision.

In [ ]:
# Create tensors in every dtype and inspect properties
dtypes = [
    ("FP32", torch.float32),
    ("FP16", torch.float16),
    ("BF16", torch.bfloat16),
    ("FP8 E4M3", torch.float8_e4m3fn),
    ("FP8 E5M2", torch.float8_e5m2),
]

x = torch.tensor([1.0, 0.1, 3.14159, 100.0, 0.0001])
print(f"Original FP32: {x.tolist()}\n")

print(f"{'Format':<12} {'Bytes/elem':<12} {'Round-trip values'}")
print("-" * 70)
for name, dtype in dtypes:
    t = x.to(dtype)
    back = t.to(torch.float32)
    print(f"{name:<12} {t.element_size():<12} {back.tolist()}")

# Range and precision for standard formats
print(f"\n{'Format':<8} {'Max':<15} {'Min Normal':<15} {'Epsilon'}")
print("-" * 55)
for name, dtype in [("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"{name:<8} {info.max:<15.6g} {info.tiny:<15.6g} {info.eps:.6g}")

## 2. Precision Loss Demo: FP32 → FP16 Round-Trip

In [ ]:
# Precision loss: FP32 → FP16 → FP32 and FP32 → BF16 → FP32
values = torch.tensor([1.0, 0.1, 3.14159265, 1e-4, 1e-7, 12345.678, 0.333333])

print("FP32 → FP16 → FP32 round-trip error:")
print(f"{'Original':<15} {'Round-trip':<15} {'Abs Error':<12} {'Rel Error'}")
print("-" * 55)
for v in values:
    orig = v.item()
    rt = v.to(torch.float16).to(torch.float32).item()
    abs_e = abs(orig - rt)
    rel_e = abs_e / abs(orig) if orig != 0 else 0
    print(f"{orig:<15.8f} {rt:<15.8f} {abs_e:<12.2e} {rel_e:.2e}")

print("\nFP32 → BF16 → FP32 round-trip error:")
print(f"{'Original':<15} {'Round-trip':<15} {'Abs Error':<12} {'Rel Error'}")
print("-" * 55)
for v in values:
    orig = v.item()
    rt = v.to(torch.bfloat16).to(torch.float32).item()
    abs_e = abs(orig - rt)
    rel_e = abs_e / abs(orig) if orig != 0 else 0
    print(f"{orig:<15.8f} {rt:<15.8f} {abs_e:<12.2e} {rel_e:.2e}")

print("\nBF16 has MORE error (fewer mantissa bits) but handles LARGE values")

## 3. BF16 vs FP16: Overflow Demonstration

In [ ]:
# BF16 vs FP16: overflow behavior
large_values = [100, 1000, 10000, 50000, 65504, 70000, 100000, 1e10, 1e20, 1e38]

print(f"{'Value':<12} {'FP16':<15} {'BF16':<15} {'FP16 OK?':<10} {'BF16 OK?'}")
print("-" * 65)
for val in large_values:
    t = torch.tensor(val, dtype=torch.float32)
    fp16 = t.to(torch.float16).to(torch.float32)
    bf16 = t.to(torch.bfloat16).to(torch.float32)
    fp16_ok = "OK" if not torch.isinf(fp16) else "OVERFLOW!"
    bf16_ok = "OK" if not torch.isinf(bf16) else "OVERFLOW!"
    print(f"{val:<12.4g} {fp16.item():<15.4g} {bf16.item():<15.4g} {fp16_ok:<10} {bf16_ok}")

print("\n→ BF16 handles values up to 3.4e38 (same as FP32)!")
print("→ FP16 overflows above 65504 — this is why GradScaler is needed")

# Why this matters: logits in LLM training
torch.manual_seed(42)
logits = torch.randn(8, 50000) * 50
print(f"\nSimulated LLM logits range: [{logits.min():.1f}, {logits.max():.1f}]")
print(f"FP16 infs: {torch.isinf(logits.to(torch.float16)).sum().item()} / {logits.numel()}")
print(f"BF16 infs: {torch.isinf(logits.to(torch.bfloat16)).sum().item()} / {logits.numel()}")

## 4. Memory Comparison

In [ ]:
# Memory usage for model parameters
num_params = 1_000_000
print(f"Memory for {num_params:,} parameters:\n")
print(f"{'Format':<12} {'Bytes/param':<12} {'Total MB':<12} {'vs FP32'}")
print("-" * 50)

fp32_bytes = num_params * 4
for name, dtype in dtypes:
    t = torch.zeros(num_params, dtype=dtype)
    total = t.element_size() * t.numel()
    print(f"{name:<12} {t.element_size():<12} {total/1e6:<12.1f} {total/fp32_bytes:.2f}x")

print(f"\n7B parameter model sizes:")
for name, dtype in dtypes:
    size_gb = 7e9 * torch.zeros(1, dtype=dtype).element_size() / 1e9
    print(f"  {name:<12}: {size_gb:.1f} GB")

## 5. AMP autocast Basics

In [ ]:
# autocast automatically casts eligible operations
linear = nn.Linear(64, 128)
norm = nn.LayerNorm(128)
x = torch.randn(8, 64)

print("Without autocast:")
out = norm(linear(x))
print(f"  Output dtype: {out.dtype}")

print("\nWith autocast('cpu', dtype=torch.bfloat16):")
with autocast('cpu', dtype=torch.bfloat16):
    out = linear(x)
    print(f"  linear output: {out.dtype}  ← cast to BF16 (matmul op)")
    out = norm(out)
    print(f"  norm output:   {out.dtype}")

print("\nCast rules (CUDA):")
print("  CAST DOWN: matmul, linear, conv, bmm")
print("  KEEP FP32: softmax, cross_entropy, layer_norm, exp, log")

# Disabling autocast selectively
print("\nSelectively disabling autocast:")
with autocast('cpu', dtype=torch.bfloat16):
    y = F.linear(x, torch.randn(128, 64))
    print(f"  autocast linear: {y.dtype}")
    with autocast('cpu', enabled=False):
        z = torch.log(torch.exp(y.float()) + 1)
        print(f"  disabled (custom op): {z.dtype}")

## 6. GradScaler Walkthrough

GradScaler prevents gradient underflow in FP16:
1. Scale loss up before backward (gradients become representable)
2. Unscale gradients before optimizer step
3. If inf/nan detected, skip step and reduce scale
4. Periodically increase scale if no overflow

**NOT needed for BF16** (same exponent range as FP32).

In [ ]:
# Why GradScaler is needed: small gradients underflow in FP16
gradient_magnitudes = [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]

print("Small gradients in FP16:")
print(f"{'Gradient':<12} {'FP16 value':<15} {'Lost?'}")
print("-" * 40)
for mag in gradient_magnitudes:
    g = torch.tensor(mag).to(torch.float16)
    lost = "UNDERFLOW" if g.item() == 0 else "ok"
    print(f"{mag:<12.1e} {g.float().item():<15.1e} {lost}")

print("\nWith GradScaler (scale=65536):")
print(f"{'Gradient':<12} {'Scaled':<12} {'FP16':<15} {'Survives?'}")
print("-" * 50)
scale = 65536.0
for mag in gradient_magnitudes:
    scaled = mag * scale
    g = torch.tensor(scaled).to(torch.float16)
    ok = "YES" if g.item() != 0 and not torch.isinf(g) else "NO"
    print(f"{mag:<12.1e} {scaled:<12.1e} {g.float().item():<15.6g} {ok}")

In [ ]:
# Scale factor dynamics
print("GradScaler scale factor dynamics:\n")
scale = 65536.0
growth_factor, backoff_factor, growth_interval = 2.0, 0.5, 5
clean_steps = 0

events = ["ok"]*5 + ["ok"]*2 + ["overflow"] + ["ok"]*5 + ["ok"]*5 + ["overflow"]

print(f"{'Step':<6} {'Event':<12} {'Scale':<12} {'Action'}")
print("-" * 45)
for step, event in enumerate(events):
    if event == "overflow":
        scale *= backoff_factor
        clean_steps = 0
        print(f"{step:<6} {'OVERFLOW':<12} {scale:<12.0f} backoff + skip")
    else:
        clean_steps += 1
        if clean_steps >= growth_interval:
            scale *= growth_factor
            clean_steps = 0
            print(f"{step:<6} {'grow':<12} {scale:<12.0f} scale increased")
        else:
            print(f"{step:<6} {'ok':<12} {scale:<12.0f} ({clean_steps}/{growth_interval})")

In [ ]:
# Complete GradScaler pattern (CUDA reference)
print("""
Complete FP16 training with GradScaler (CUDA):

    model = Model().cuda()
    optimizer = AdamW(model.parameters(), lr=1e-4)
    scaler = GradScaler('cuda')

    for data, target in dataloader:
        optimizer.zero_grad()
        with autocast('cuda', dtype=torch.float16):
            loss = criterion(model(data.cuda()), target.cuda())
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
""")

## 7. Complete Training: FP32 vs FP16 vs BF16 Comparison

In [ ]:
# Model and data
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.LayerNorm(128),
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
X = torch.randn(512, 64)
y = torch.randint(0, 10, (512,))
criterion = nn.CrossEntropyLoss()

# FP32 training
torch.manual_seed(42)
model_fp32 = MLP()
opt = torch.optim.Adam(model_fp32.parameters(), lr=1e-3)
losses_fp32 = []
for epoch in range(30):
    opt.zero_grad()
    loss = criterion(model_fp32(X), y)
    loss.backward()
    opt.step()
    losses_fp32.append(loss.item())

# BF16 training
torch.manual_seed(42)
model_bf16 = MLP()
opt = torch.optim.Adam(model_bf16.parameters(), lr=1e-3)
losses_bf16 = []
for epoch in range(30):
    opt.zero_grad()
    with autocast('cpu', dtype=torch.bfloat16):
        loss = criterion(model_bf16(X), y)
    loss.backward()
    opt.step()
    losses_bf16.append(loss.item())

print(f"{'Epoch':<8} {'FP32':<12} {'BF16'}")
print("-" * 35)
for i in range(0, 30, 5):
    print(f"{i:<8} {losses_fp32[i]:<12.4f} {losses_bf16[i]:.4f}")
print(f"\nFP32 final: {losses_fp32[-1]:.4f}")
print(f"BF16 final: {losses_bf16[-1]:.4f}")
print("\n→ Both converge to similar loss! BF16 is 2-3x faster on GPU.")

## 8. FP8 Overview and Scaled Matmul

In [ ]:
# FP8 formats
print("FP8 E4M3 (forward pass — more precision, range ±448):")
test_vals = torch.tensor([0.5, 1.0, 10.0, 100.0, 400.0, 448.0, 500.0])
for v in test_vals:
    fp8 = v.unsqueeze(0).to(torch.float8_e4m3fn).to(torch.float32).item()
    status = '(saturated)' if v.item() > 448 else ''
    print(f"  {v.item():>8.1f} → {fp8:>8.2f}  {status}")

print("\nFP8 E5M2 (backward pass — more range, ±57344):")
test_vals = torch.tensor([0.5, 1.0, 100.0, 1000.0, 10000.0, 50000.0])
for v in test_vals:
    fp8 = v.unsqueeze(0).to(torch.float8_e5m2).to(torch.float32).item()
    print(f"  {v.item():>10.1f} → {fp8:>10.2f}")

In [ ]:
# FP8 scaled matmul concept
torch.manual_seed(42)
A = torch.randn(64, 128) * 3.0
B = torch.randn(128, 256) * 2.0
ref = A @ B

# Per-tensor scaling + FP8 quantization
e4m3_max = 448.0
scale_A = e4m3_max / A.abs().max()
scale_B = e4m3_max / B.abs().max()
A_fp8 = (A * scale_A).to(torch.float8_e4m3fn).to(torch.float32)
B_fp8 = (B * scale_B).to(torch.float8_e4m3fn).to(torch.float32)
result_fp8 = (A_fp8 @ B_fp8) / (scale_A * scale_B)

# BF16 comparison
result_bf16 = A.bfloat16().float() @ B.bfloat16().float()

print(f"Matmul: [64×128] @ [128×256]")
print(f"  FP8 mean error:  {(ref - result_fp8).abs().mean():.6f}")
print(f"  BF16 mean error: {(ref - result_bf16).abs().mean():.6f}")
print(f"  FP8/BF16 ratio:  {(ref - result_fp8).abs().mean() / (ref - result_bf16).abs().mean():.1f}x less accurate")
print(f"  But ~2x faster on H100 tensor cores!")

print("\nCUDA API: torch._scaled_mm(a_fp8, b_fp8.t(), scale_a=..., scale_b=..., out_dtype=torch.bfloat16)")

## 9. Mixed Precision with FSDP2

In [ ]:
# FSDP2 MixedPrecisionPolicy pattern
print("""
FSDP2 Mixed Precision:

    from torch.distributed._composable.fsdp import fully_shard, MixedPrecisionPolicy

    mp_policy = MixedPrecisionPolicy(
        param_dtype=torch.bfloat16,    # Compute in BF16
        reduce_dtype=torch.float32,    # All-reduce in FP32
    )

    for block in model.layers:
        fully_shard(block, mp_policy=mp_policy)
    fully_shard(model, mp_policy=mp_policy)

    # No autocast or GradScaler needed!
    for data, target in dataloader:
        loss = criterion(model(data), target)
        loss.backward()
        optimizer.step()
""")

# Why FP32 reduce matters
torch.manual_seed(42)
grads = [torch.randn(100) * 1e-5 for _ in range(8)]
avg_fp32 = torch.stack(grads).mean(dim=0)
avg_bf16 = torch.stack([g.bfloat16() for g in grads]).mean(dim=0).float()
error = (avg_fp32 - avg_bf16).abs()
print(f"Gradient averaging across 8 GPUs (magnitude ~1e-5):")
print(f"  Error from BF16 reduce: {error.mean():.2e}")
print(f"  Signal lost: {(error.mean() / avg_fp32.abs().mean()):.0%}")
print(f"  → Use FP32 reduce to preserve gradient information!")

## 10. Quantization Error Analysis

In [ ]:
# Error distribution for typical neural network activations
torch.manual_seed(42)
activations = torch.randn(10000)  # N(0,1) post-normalization

formats = [
    ("FP16", torch.float16),
    ("BF16", torch.bfloat16),
    ("FP8 E4M3", torch.float8_e4m3fn),
    ("FP8 E5M2", torch.float8_e5m2),
]

print(f"Quantization error on 10K values from N(0,1):\n")
print(f"{'Format':<12} {'Mean Error':<12} {'Max Error':<12} {'RMSE'}")
print("-" * 50)
for name, dtype in formats:
    cast = activations.to(dtype).to(torch.float32)
    err = (activations - cast).abs()
    rmse = ((activations - cast)**2).mean().sqrt()
    print(f"{name:<12} {err.mean():<12.6f} {err.max():<12.6f} {rmse:.6f}")

## 11. Mixed Precision with torch.compile

In [ ]:
# torch.compile composes with autocast
torch.manual_seed(42)
model_c = MLP()
compiled = torch.compile(model_c, backend='eager')
opt = torch.optim.Adam(model_c.parameters(), lr=1e-3)
losses_compiled = []

for epoch in range(30):
    opt.zero_grad()
    with autocast('cpu', dtype=torch.bfloat16):
        loss = criterion(compiled(X), y)
    loss.backward()
    opt.step()
    losses_compiled.append(loss.item())

print(f"Compiled + BF16 final loss: {losses_compiled[-1]:.4f}")
print(f"Non-compiled BF16 final loss: {losses_bf16[-1]:.4f}")
print(f"\ntorch.compile optimizes mixed precision:")
print(f"  • Fuses cast operations (fewer kernel launches)")
print(f"  • Eliminates redundant dtype conversions")
print(f"  • Optimizes accumulation patterns")
print(f"\ntorch.set_float32_matmul_precision options:")
print(f"  'highest' — Pure FP32 (slowest)")
print(f"  'high'    — TF32 on Ampere+ (default, good balance)")
print(f"  'medium'  — Reduced accumulators (fastest)")

## 12. Numerical Stability Checklist

| Issue | Symptom | Solution |
|-------|---------|----------|
| Gradient underflow | Loss plateaus, zero gradients | GradScaler (FP16) or switch to BF16 |
| Loss explosion | Loss → inf/NaN | Gradient clipping, reduce LR, use BF16 |
| NaN in softmax | NaN outputs | Keep softmax in FP32 (autocast default) |
| Accumulation error | Reduced accuracy | FP32 accumulators (default for matmul) |
| Optimizer divergence | Unstable after many steps | Keep optimizer states in FP32 (default) |

In [ ]:
# Precision health check utility
def check_health(model, loss_val, step):
    """Quick health check for mixed precision training."""
    issues = []
    if torch.isnan(torch.tensor(loss_val)): issues.append("Loss is NaN")
    if torch.isinf(torch.tensor(loss_val)): issues.append("Loss is inf")
    
    total, zeros = 0, 0
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.numel()
            zeros += (p.grad == 0).sum().item()
    
    if total > 0 and zeros/total > 0.5:
        issues.append(f"{zeros/total:.0%} gradients zero (underflow?)")
    
    status = f"ISSUES: {'; '.join(issues)}" if issues else f"Healthy (loss={loss_val:.4f})"
    print(f"  Step {step}: {status}")

# Demo
torch.manual_seed(42)
model_h = MLP()
opt = torch.optim.Adam(model_h.parameters(), lr=1e-3)
for step in range(5):
    opt.zero_grad()
    loss = criterion(model_h(X[:64]), y[:64])
    loss.backward()
    check_health(model_h, loss.item(), step)
    opt.step()

## 13. Decision Guide

| Precision | Speed | Memory | GradScaler? | GPU Requirement |
|-----------|-------|--------|-------------|------------------|
| FP32 | 1× | 4B/param | No | Any |
| TF32 | ~2× | 4B/param | No | Ampere+ (automatic) |
| FP16 | 2-3× | 2B/param | **Yes** | Volta+ |
| BF16 | 2-3× | 2B/param | No | Ampere+ |
| FP8 | 4-6× | 1B/param | Scaling needed | H100+ |

In [ ]:
print("""
Which Precision to Use?

    H100           → BF16 + torch.compile (consider FP8 for large models)
    A100           → BF16 + torch.compile
    V100 / T4      → FP16 + GradScaler
    CPU (Intel)    → BF16 autocast (limited ops)
    CPU (general)  → FP32

Always:
  • Keep optimizer states in FP32
  • Use gradient clipping (max_norm=1.0)
  • Monitor for NaN/inf in early training
""")

## Exercise

**Task:** Train a model in FP32, FP16 (simulated), and BF16 — compare final loss and training speed.

1. Create a larger model (3+ layers, 256 hidden dim)
2. Generate synthetic data (2000+ samples, 20 classes)
3. Train each for 50 epochs
4. Compare: final loss, convergence speed
5. Bonus: Add gradient clipping and observe its effect

In [ ]:
# Exercise starter code
class LargerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.LayerNorm(256),
            nn.Linear(256, 256), nn.ReLU(), nn.LayerNorm(256),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 20)
        )
    def forward(self, x):
        return self.net(x)

# TODO: Generate data, train in FP32 and BF16, compare losses
# torch.manual_seed(42)
# X_ex = torch.randn(2000, 128)
# y_ex = torch.randint(0, 20, (2000,))
# ...

## Key Takeaways

1. **BF16 is the default for modern training** — same range as FP32, no GradScaler needed, 2-3x faster on Ampere+

2. **FP16 requires GradScaler** — limited range (max 65504) causes gradient underflow without loss scaling

3. **FP8 is the cutting edge** — 2x over BF16 on H100, but requires per-tensor scaling

4. **autocast handles per-op decisions** — matmul/conv cast down, softmax/norm stay in FP32

5. **torch.compile optimizes precision** — fuses casts, eliminates redundant conversions

6. **FSDP2 MixedPrecisionPolicy** — compute in BF16, reduce in FP32 for distributed stability

7. **Always keep optimizer in FP32** — Adam's running averages need full precision

8. **Monitor training health** — check for zero gradients (underflow) and inf/NaN (overflow)